In [13]:
pip install pandas requests

Note: you may need to restart the kernel to use updated packages.


In [24]:
import pandas as pd
import requests
import time
import collections
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---------------------------------------------------------
# PLATFORM SPECIFIC HANDLERS
# ---------------------------------------------------------

def fetch_greenhouse_jobs(company, api_url):
    """Greenhouse Public API."""
    jobs = []
    try:
        data = requests.get(api_url, timeout=15).json()
        for job in data.get('jobs', []):
            jobs.append({
                'Company': company, 
                'Job Title': job.get('title'),
                'Location': job.get('location', {}).get('name', 'Unknown'),
                'Department': job.get('departments', [{'name': ''}])[0].get('name', ''),
                'Job URL': job.get('absolute_url'), 
                'Updated At': job.get('updated_at')
            })
    except Exception as e: print(f"Error {company}: {e}")
    return jobs

def fetch_eightfold_jobs(company, api_url):
    """Eightfold.ai with Pagination."""
    jobs, offset = [], 0
    try:
        while True:
            url = f"{api_url}&start={offset}" if "?" in api_url else f"{api_url}?start={offset}"
            data = requests.get(url, timeout=15).json()
            positions = data.get('positions', [])
            if not positions: break
            for pos in positions:
                jobs.append({
                    'Company': company, 
                    'Job Title': pos.get('name'),
                    'Location': pos.get('location', 'Unknown'), 
                    'Job URL': pos.get('canonicalPositionUrl')
                })
            offset += 10
            time.sleep(0.5)
    except Exception as e: print(f"Error {company}: {e}")
    return jobs

def fetch_teamtailor_jobs(company, api_url):
    """Teamtailor API (Requires specific headers)."""
    jobs = []
    headers = {'Accept': 'application/vnd.api+json', 'X-Api-Version': '20210218'}
    try:
        data = requests.get(api_url, headers=headers, timeout=15).json()
        for job in data.get('data', []):
            attr = job.get('attributes', {})
            jobs.append({
                'Company': company, 
                'Job Title': attr.get('title'),
                'Location': attr.get('remote-status') or 'Office',
                'Job URL': job.get('links', {}).get('careersite-job-url'),
                'Updated At': attr.get('created-at')
            })
    except Exception as e: print(f"Error {company}: {e}")
    return jobs

def fetch_gatsby_i18n_jobs(company, api_url):
    """Gatsby Static JSON (AQUMON style)."""
    jobs = []
    try:
        data = requests.get(api_url, timeout=15).json()
        extracted = collections.defaultdict(dict)
        def find_keys(node):
            if isinstance(node, dict):
                for k, v in node.items():
                    if k.startswith('career.job.jobsInfo.'):
                        p = k.split('.')
                        if len(p) >= 5: extracted[p[3]][p[4]] = v
                    else: find_keys(v)
            elif isinstance(node, list):
                for item in node: find_keys(item)
        find_keys(data)
        for slug, j in extracted.items():
            if 'hong kong' in str(j.get('location', '')).lower():
                jobs.append({
                    'Company': company, 
                    'Job Title': j.get('title'),
                    'Location': j.get('location'), 
                    'Department': j.get('department'),
                    'Job URL': f"https://www.aqumon.com/about/career/{slug}"
                })
    except Exception as e: print(f"Error {company}: {e}")
    return jobs

def fetch_oracle_hcm_jobs(company, api_url, site="CX_3002", loc_id=None):
    """Oracle Cloud HCM (Handles Goldman Sachs and JPM)."""
    jobs, offset, limit = [], 0, 25
    host = f"{urlparse(api_url).scheme}://{urlparse(api_url).netloc}"
    finder = f"findReqs;siteNumber={site},sortBy=POSTING_DATES_DESC"
    if loc_id: finder = f"findReqs;siteNumber={site},locationId={loc_id},offset={offset}"
    
    try:
        while True:
            params = {"onlyData": "true", "expand": "requisitionList", "finder": finder, "limit": limit, "offset": offset}
            data = requests.get(api_url, params=params, timeout=15).json()
            if 'items' not in data or not data['items']: break
            root = data['items'][0]
            batch = root.get('requisitionList', [])
            if not batch: break
            for j in batch:
                jid = j.get('Id')
                jobs.append({
                    'Company': company, 
                    'Job Title': j.get('Title'),
                    'Location': j.get('PrimaryLocation', 'Unknown'), 
                    'Updated At': j.get('PostedDate'),
                    'Job URL': f"{host}/hcmUI/CandidateExperience/en/sites/{site}/job/{jid}"
                })
            if offset + len(batch) >= root.get('TotalJobsCount', 0): break
            offset += limit
            time.sleep(1)
    except Exception as e: print(f"Error {company}: {e}")
    return jobs

def fetch_hkex_workday_jobs(company):
    """Workday with CSRF (HKEX)."""
    jobs = []
    base = "https://hkex.wd3.myworkdayjobs.com/HKEXCareerPage"
    api = "https://hkex.wd3.myworkdayjobs.com/wday/cxs/hkex/HKEXCareerPage/jobs"
    s = requests.Session()
    try:
        s.get(base, timeout=15)
        csrf = s.cookies.get("CALYPSO_CSRF_TOKEN")
        offset, total = 0, None
        while True:
            payload = {"appliedFacets": {}, "limit": 20, "offset": offset, "searchText": ""}
            r = s.post(api, json=payload, headers={"x-calypso-csrf-token": csrf}, timeout=15).json()
            if total is None: total = r.get("total", 0)
            batch = r.get("jobPostings", [])
            if not batch: break
            for j in batch:
                jobs.append({
                    'Company': company, 
                    'Job Title': j.get('title'),
                    'Location': j.get('locationsText', 'Unknown'), 
                    'Updated At': j.get('postedOn'),
                    'Job URL': f"{base}{j.get('externalPath')}"
                })
            if len(jobs) >= total: break
            offset += 20
            time.sleep(1)
    except Exception as e: print(f"Error {company}: {e}")
    return jobs

def fetch_hkma_ssr_jobs(company, url):
    """Custom SSR Table parsing (HKMA)."""
    jobs = []
    try:
        r = requests.get(url, timeout=15)
        soup = BeautifulSoup(r.text, 'html.parser')
        table = soup.find('table')
        if table:
            for row in table.find_all('tr')[1:]:
                cols = row.find_all('td')
                if len(cols) >= 2:
                    link = cols[0].find('a')
                    jobs.append({
                        'Company': company, 
                        'Job Title': cols[0].get_text(strip=True),
                        'Location': 'Hong Kong', 
                        'Job URL': f"https://www.hkma.gov.hk{link['href']}" if link else url
                    })
    except Exception as e: print(f"Error {company}: {e}")
    return jobs

# ---------------------------------------------------------
# THE MERGED GENERIC FALLBACK (WITH RETRY & TIMEOUT)
# ---------------------------------------------------------

def fetch_generic_jobs(company_name, api_url):
    """
    Your old, highly effective fallback parser for Lever, Ashby, and unknowns.
    Upgraded with a 30s timeout and a robust retry strategy so Airwallex won't fail.
    """
    jobs_list = []
    
    # Retry mechanism allows Ashby to 'breathe' if it hangs
    retry_strategy = Retry(
        total=3,
        backoff_factor=2, 
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session = requests.Session()
    session.mount("https://", adapter)
    
    try:
        # Increased timeout to 30s for heavy endpoints like Airwallex
        response = session.get(api_url, timeout=30)
        
        if response.status_code == 200:
            data = response.json()
            # If it's a direct list of jobs (e.g., Lever often returns a list directly)
            if isinstance(data, list):
                jobs = data
            elif isinstance(data, dict):
                # Guess where the jobs are stored
                if 'jobs' in data: jobs = data['jobs']
                elif 'data' in data: jobs = data['data']
                else: jobs = []
            
            for job in jobs:
                jobs_list.append({
                    'Company': company_name,
                    'Job Title': job.get('text') or job.get('title') or job.get('name', 'Unknown'),
                    'Location': str(job.get('location') or job.get('categories', {}).get('location', 'Unknown')),
                    # Added jobUrl natively to support Ashby seamlessly
                    'Job URL': job.get('hostedUrl') or job.get('applyUrl') or job.get('jobUrl') or job.get('absolute_url', 'Unknown'),
                    'Department': str(job.get('categories', {}).get('team', '')),
                    'Updated At': job.get('createdAt') or job.get('updated_at') or job.get('publishedAt', '')
                })
        else:
            print(f"[{company_name}] Failed to fetch. Status Code: {response.status_code}")
            
    except Exception as e:
        print(f"Error parsing generic JSON for {company_name}: {e}")
        
    return jobs_list

# ---------------------------------------------------------
# MAIN DISPATCHER
# ---------------------------------------------------------

def main():
    input_csv = "target_companies_api.csv"
    try:
        df = pd.read_csv(input_csv)
    except FileNotFoundError:
        print(f"Error: Could not find {input_csv}.")
        return

    valid_apis = df[df['API Endpoint'].notna() & (df['API Endpoint'] != 'N/A') & (df['API Endpoint'] != '')]
    print(f"Found {len(valid_apis)} companies with valid API endpoints.\n")
    
    master_list = []

    for _, row in valid_apis.iterrows():
        c = row['Company']
        p = str(row['Platform']).lower()
        u = str(row['API Endpoint'])
        
        print(f"Processing {c}...")

        # Explicit High-Complexity APIs
        if c == "HKEX": results = fetch_hkex_workday_jobs(c)
        elif c == "HKMA": results = fetch_hkma_ssr_jobs(c, u)
        elif c == "JPM": results = fetch_oracle_hcm_jobs(c, "https://jpmc.fa.oraclecloud.com/hcmRestApi/resources/latest/recruitingCEJobRequisitions", site="CX_1001", loc_id="300000000289330")
        elif 'greenhouse' in p: results = fetch_greenhouse_jobs(c, u)
        elif 'teamtailor' in p: results = fetch_teamtailor_jobs(c, u)
        elif 'eightfold' in p: results = fetch_eightfold_jobs(c, u)
        elif 'gatsby' in p: results = fetch_gatsby_i18n_jobs(c, u)
        elif 'oracle cloud' in p: results = fetch_oracle_hcm_jobs(c, u)
        
        # Merge Old Logic: Catch Lever, Ashby, and any other standard/unknown JSON APIs
        elif 'ashby' in p or 'lever' in p: 
            results = fetch_generic_jobs(c, u)
        else:
            results = fetch_generic_jobs(c, u) # Final Fallback for unhandled platforms
            
        print(f"-> Found {len(results)} jobs.")
        master_list.extend(results)
        
        time.sleep(1) # Be polite to APIs

    if master_list:
        output_df = pd.DataFrame(master_list)
        # Reorder to guarantee nice CSV headers based on your old code
        columns = ['Company', 'Job Title', 'Location', 'Department', 'Updated At', 'Job URL']
        for col in columns:
            if col not in output_df.columns: output_df[col] = None
        output_df = output_df.reindex(columns=columns)
        
        output_df.to_csv("master_jobs_list.csv", index=False)
        print(f"\n✅ Done. Total jobs collected: {len(master_list)}")
    else:
        print("\nNo jobs could be extracted.")

if __name__ == "__main__":
    main()

Found 13 companies with valid API endpoints.

Processing Crypto.com...
-> Found 43 jobs.
Processing OKX...
-> Found 305 jobs.
Processing Airwallex...
-> Found 622 jobs.
Processing Reap...
-> Found 10 jobs.
Processing AQUMON...
-> Found 20 jobs.
Processing Goldman Sachs...


KeyboardInterrupt: 

In [22]:
jobs_csv = "master_jobs_list.csv"
try:
    df_jobs = pd.read_csv(jobs_csv)
except FileNotFoundError:
    print(f"Error: Could not find {jobs_csv}.")
    
filtered_df = df_jobs[df_jobs['Location'].str.lower().str.contains('hong', na=False) | df_jobs['Location'].str.lower().str.contains('hk', na=False)]
output_file = "master_jobs_list_filtered.csv"
filtered_df.to_csv(output_file, index=False)